# Calibration interactive des plages HSV

**PFA — Mohamed GHARBI**

Trois outils pour calibrer la segmentation couleur sur ta carte :

- **Outil 0** — détection automatique du cadre cartographique
- **Outil 1** — échantillonnage par coordonnées
- **Outil 2** — sliders interactifs en direct

Compatible **Colab** (GPU T4) ou **local Anaconda**.

## Setup Colab (sans effet en local)

In [ ]:
REPO_URL  = 'https://github.com/Mohamed-GHARBI/pfa.git'    # adapte
BRANCH    = 'main'
USE_DRIVE = False

import sys, subprocess, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Environnement :', 'Google Colab' if IN_COLAB else 'Local')

if IN_COLAB:
    print('\nInstallation des dependances...')
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
        'opencv-python==4.10.0.84', 'scikit-image>=0.22',
        'rasterio>=1.3', 'shapely>=2.0', 'geopandas>=0.14',
        'pyogrio', 'fiona', 'ipywidgets>=8.1'])
    print('OK')

    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = Path('/content/drive/MyDrive/pfa')
    else:
        PROJECT_ROOT = Path('/content/pfa')
        if not (PROJECT_ROOT / 'pipeline').exists():
            subprocess.check_call(['git', 'clone', '--depth', '1',
                                     '--branch', BRANCH, REPO_URL,
                                     str(PROJECT_ROOT)])
else:
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT : {PROJECT_ROOT}')

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, IntSlider, Dropdown, fixed

from pipeline import preprocessing as prep
from pipeline import color_segmentation as colseg

INPUT_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'carte_test.png')
print('Carte :', INPUT_PATH, '| existe :', os.path.exists(INPUT_PATH))

## Outil 0 — Détection automatique du cadre

In [ ]:
image_full = prep.load_image(INPUT_PATH)
image_full = prep.downscale_if_too_large(image_full, max_dimension=2400)
image_full = prep.denoise(image_full)
H_full, W_full = image_full.shape[:2]
auto_bbox = prep.detect_map_frame(image_full)
x1, y1, x2, y2 = auto_bbox

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(prep.to_rgb(image_full))
ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=3, edgecolor='lime', facecolor='none',
                                  label='Cadre detecte'))
ax.set_title(f'Image {W_full}x{H_full}  --  cadre ({x1},{y1})->({x2},{y2})')
ax.legend(loc='upper right'); plt.tight_layout(); plt.show()

In [ ]:
MANUAL_BBOX = None  # ou (180, 220, 2400, 1800)
image_bgr, image_hsv, used_bbox = prep.preprocess_with_crop(
    INPUT_PATH, auto_crop=True, manual_bbox=MANUAL_BBOX, max_dimension=2400)
image_rgb = prep.to_rgb(image_bgr)
H, W = image_rgb.shape[:2]
print(f'Carte recadree : {W} x {H} (bbox : {used_bbox})')

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(image_rgb)
ax.set_title(f'Carte recadree ({W}x{H})')
ax.set_xticks(np.arange(0, W, max(W // 20, 1)))
ax.set_yticks(np.arange(0, H, max(H // 15, 1)))
ax.grid(True, alpha=0.3, color='cyan', linestyle='--')
plt.tight_layout(); plt.show()

## Outil 1 — Échantillonnage par coordonnées

Repère 3-5 pixels typiques par catégorie sur la carte ci-dessus, mets leurs (x, y) ci-dessous.

In [ ]:
WATER_SAMPLES      = [(1500, 200), (1530, 220), (1480, 260), (1550, 240)]
VEGETATION_SAMPLES = [(800, 600),  (820, 620),  (810, 640)]
CONTOURS_SAMPLES   = [(1200, 800), (1180, 780), (1220, 820)]
RED_ROAD_SAMPLES   = [(900, 500),  (920, 480),  (880, 520)]

categories = {'water':WATER_SAMPLES, 'vegetation':VEGETATION_SAMPLES,
              'contours':CONTOURS_SAMPLES, 'red_roads':RED_ROAD_SAMPLES}

for name, points in categories.items():
    valid = [(x, y) for (x, y) in points if 0 <= x < W and 0 <= y < H]
    if not valid:
        print(f'{name:12s} -> pas de points valides'); continue
    samples = colseg.sample_hsv_at(image_hsv, valid)
    rng = colseg.suggest_range_from_samples(samples)
    print(f'{name:12s} HSV : {samples.tolist()}')
    print(f'             -> {rng}\n')

## Outil 2 — Sliders interactifs

In [ ]:
def preview_mask(category, h_min, h_max, s_min, s_max, v_min, v_max,
                  density_window, density_threshold, apply_density):
    rng = colseg.HSVRange(h_min=h_min, s_min=s_min, v_min=v_min,
                           h_max=h_max, s_max=s_max, v_max=v_max)
    mask = colseg.mask_from_range(image_hsv, rng)
    mask = colseg.clean_mask(mask, open_kernel=3, close_kernel=5, min_area=80)
    if apply_density:
        mask = colseg.density_filter(mask, window=density_window,
                                       min_density=density_threshold/100)
        mask = colseg.clean_mask(mask, open_kernel=3, close_kernel=5, min_area=150)
    coverage = colseg.coverage_percent(mask)
    overlay = image_rgb.copy()
    color = {'water':(0,120,255), 'vegetation':(0,200,0),
             'contours':(160,80,30), 'red_roads':(255,0,0)}.get(category, (255,255,0))
    overlay[mask > 0] = color
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    axes[0].imshow(image_rgb);  axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(mask, cmap='gray'); axes[1].set_title(f'Masque ({coverage:.2f}%)'); axes[1].axis('off')
    axes[2].imshow(overlay);    axes[2].set_title('Overlay');  axes[2].axis('off')
    plt.tight_layout(); plt.show()
    print(f'\nA COPIER dans color_segmentation.py :')
    print(f'  "{category}": HSVRange(h_min={h_min}, s_min={s_min}, v_min={v_min}, '
          f'h_max={h_max}, s_max={s_max}, v_max={v_max}),')

interact(preview_mask,
         category=Dropdown(options=['water','vegetation','contours','red_roads'], value='water'),
         h_min=IntSlider(min=0, max=179, value=95),  h_max=IntSlider(min=0, max=179, value=130),
         s_min=IntSlider(min=0, max=255, value=60),  s_max=IntSlider(min=0, max=255, value=255),
         v_min=IntSlider(min=0, max=255, value=60),  v_max=IntSlider(min=0, max=255, value=255),
         density_window=IntSlider(min=5, max=61, step=2, value=25),
         density_threshold=IntSlider(min=5, max=80, value=25),
         apply_density=fixed(True));

## Conseils par couche

- **water** : H 95-130, S min ≥ 60, density_filter actif.
- **vegetation** : H 35-85, density_filter actif.
- **contours** : H 8-22, **pas** de density_filter.
- **red_roads** : 2 plages (RED_RANGE_LOW + HIGH), S min ≥ 100.

Une fois calibré : édite `pipeline/color_segmentation.py` → `DEFAULT_RANGES`, puis relance le notebook 01.